# 대학알리미 재적학생수 기반 EDSS OpenID 후보 검산

## tl;dr

- 대학알리미 학교 ID 380개 중 245개가 2023·2024년 모두 학교구분·지역·본분교·재적학생수로 하나의 같은 EDSS OpenID에 정확히 일치했다.
- 245개 후보는 서로 다른 OpenID 245개와 양방향 1:1이며 취업통계 학교·연도 482개에 연결됐다.
- 기존 학과서명 후보와 겹치는 24개 학교·연도는 24개 모두 동의했고 충돌은 0개였다.
- 공식 교차표가 아니므로 canonical `개방ID` 대입은 0건이다.

## Context & Methods

취업통계 2023–2024년에는 학교명이 있지만 `개방ID`가 없다. 대학알리미의 학교 ID·학교명·재적학생수와 EDSS `0101 고등교육학교개황`의 익명 OpenID·재적학생수를 독립적으로 대조한다.

### Key Assumptions

1. 재적학생수는 근사값이 아니라 같은 연도의 정확한 문자열 값만 비교한다.
2. 학교구분·허용 지역·본분교 문맥 안에서 각 연도에 OpenID가 하나일 때만 사용한다.
3. 2023년과 2024년이 같은 OpenID를 선택하고 역방향도 유일해야 한다.
4. 결과는 검토 후보이며 공식 학교명–OpenID 교차표가 아니다.

## Data

- `data/metadata/academyinfo_enrollment_collection.json`: 대학알리미 수집 품질 요약
- `data/metadata/edss_academyinfo_open_id_match.json`: 전체 매칭 품질 요약
- `data/metadata/edss_academyinfo_open_id_candidates.csv`: 학교 단위 후보
- `data/metadata/edss_employment_enrollment_open_id_candidates.csv`: 취업 학교·연도 후보

In [1]:
import csv
import json
from collections import Counter
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "data" / "metadata").exists():
    repo_root = repo_root.parent

collection = json.loads((repo_root / "data/metadata/academyinfo_enrollment_collection.json").read_text(encoding="utf-8"))
match = json.loads((repo_root / "data/metadata/edss_academyinfo_open_id_match.json").read_text(encoding="utf-8"))

with (repo_root / "data/metadata/edss_academyinfo_open_id_candidates.csv").open(encoding="utf-8", newline="") as handle:
    public_candidates = list(csv.DictReader(handle))
with (repo_root / "data/metadata/edss_employment_enrollment_open_id_candidates.csv").open(encoding="utf-8", newline="") as handle:
    employment_candidates = list(csv.DictReader(handle))

print({
    "collection_status": collection["status"],
    "raw_file_count": collection["raw_file_count"],
    "public_rows": collection["aggregate_row_count"],
    "match_status": match["status"],
    "public_candidate_rows": len(public_candidates),
    "employment_candidate_rows": len(employment_candidates),
})

{'collection_status': 'complete', 'raw_file_count': 756, 'public_rows': 754, 'match_status': 'review_required', 'public_candidate_rows': 380, 'employment_candidate_rows': 482}


## Results

In [2]:
candidate_rows = [row for row in public_candidates if row["candidate_open_id"]]
candidate_open_ids = {row["candidate_open_id"] for row in candidate_rows}
employment_keys = {(row["_panel_year"], row["_school_identity_key"]) for row in employment_candidates}
signature_rows = [row for row in employment_candidates if row["cross_validation_status"] != "not_available"]

checks = {
    "candidate_school_count": len(candidate_rows),
    "candidate_distinct_open_id_count": len(candidate_open_ids),
    "employment_school_year_count": len(employment_candidates),
    "employment_key_count": len(employment_keys),
    "signature_agreement_count": sum(row["cross_validation_status"] == "department_signature_agreement" for row in signature_rows),
    "signature_conflict_count": sum(row["cross_validation_status"] == "department_signature_conflict" for row in signature_rows),
    "canonical_imputed_rows": match["safety"]["canonical_open_id_imputed_row_count"],
}

assert checks == {
    "candidate_school_count": 245,
    "candidate_distinct_open_id_count": 245,
    "employment_school_year_count": 482,
    "employment_key_count": 482,
    "signature_agreement_count": 24,
    "signature_conflict_count": 0,
    "canonical_imputed_rows": 0,
}
checks

{'candidate_school_count': 245,
 'candidate_distinct_open_id_count': 245,
 'employment_school_year_count': 482,
 'employment_key_count': 482,
 'signature_agreement_count': 24,
 'signature_conflict_count': 0,
 'canonical_imputed_rows': 0}

In [3]:
status_counts = Counter(row["resolution_status"] for row in public_candidates)
agreement_examples = [
    {
        "year": row["_panel_year"],
        "school": row["학교명"],
        "candidate_open_id": row["candidate_open_id"],
        "enrollment": row["academyinfo_enrollment"],
    }
    for row in employment_candidates
    if row["cross_validation_status"] == "department_signature_agreement"
][:8]

print("public resolution status:", dict(status_counts))
print("department-signature agreement examples:")
for row in agreement_examples:
    print(row)

public resolution status: {'unresolved_two_year_exact_match_not_unique': 89, 'unresolved_missing_two_year_numeric_enrollment': 46, 'candidate_two_year_exact_enrollment': 245}
department-signature agreement examples:
{'year': '2023', 'school': '광주교육대학교', 'candidate_open_id': '7730154871', 'enrollment': '1335'}
{'year': '2023', 'school': '대구교육대학교', 'candidate_open_id': '2183489322', 'enrollment': '1602'}
{'year': '2023', 'school': '부산교육대학교', 'candidate_open_id': '6089915394', 'enrollment': '1499'}
{'year': '2023', 'school': '서울교육대학교', 'candidate_open_id': '8185494184', 'enrollment': '1497'}
{'year': '2023', 'school': '서울신학대학교', 'candidate_open_id': '6105046808', 'enrollment': '2456'}
{'year': '2023', 'school': '서울장신대학교', 'candidate_open_id': '4364512026', 'enrollment': '379'}
{'year': '2023', 'school': '영남신학대학교', 'candidate_open_id': '8771178662', 'enrollment': '484'}
{'year': '2023', 'school': '장로회신학대학교', 'candidate_open_id': '3061540484', 'enrollment': '791'}


## Takeaways

두 연도 재적학생수와 학교 문맥을 함께 사용하면 245개 학교에서 양방향 1:1 OpenID 후보를 재현할 수 있다. 독립 학과서명과 겹치는 24개가 모두 동의해 후보 규칙의 방향성은 지지된다. 다만 대학원·빈 응답·학교명 변경·캠퍼스 통합 범위는 남아 있고, 공식 교차표가 없으므로 이 후보를 원본이나 canonical 패널 키에 대입하지 않는다.